# BioLangFusion Phase A: fungal expression

**Question.** On the supplied fungal-expression split, how do BioLangFusion's fusion methods compare with their constituent single encoders across three training seeds?

Step 3 builds and checks the models. This Stage 4 notebook defines one comparison and inspects completed results from [Step 3](../stage3/03_blf_build_check.ipynb). It checks whether the records belong together, selects models using validation scores, and prepares tables and figures only when the planned grid is complete and its input headers have been independently checked. Training is disabled; no model fitting, encoder loading, cache extraction, or result-file writing is included.

Each row supplies a sequence and an expression target. DNA, RNA, and protein inputs are derived views of that sequence, not three independently measured assays. A fusion gain would show predictive benefit under this evaluation; it would not by itself establish biological alignment or a causal mechanism.

This is the fungal-expression part of Phase A, not the full five-benchmark reproduction. The research plan places reproduction before interpreting gains and introducing more complex designs: [Gantt workbook](../../docs/Research_Project_Plan_Gantt.xlsx), `Details` items 17 and 19.

---

## Setup

Run from this notebook's folder or another folder inside the repository, using the project's notebook environment. The path block locates `Code/shared_code/`; it does not depend on a personal drive path.

The imports reuse the maintained implementation:

| Source | Role here |
| --- | --- |
| [`datasets.py`](../shared_code/datasets.py) | Read the same benchmark rows and supplied splits as Step 3. |
| [`train.py`](../shared_code/fusion/train.py) | Use `TrainConfig` and `Result.load`; never call the training loop. |
| [`blf.py`](../shared_code/fusion/blf.py) | Read the modality-to-encoder mapping. |
| [`token_embeddings.py`](../shared_code/fusion/token_embeddings.py) | Reuse input and file fingerprints, without opening embedding arrays. |

Importing these modules requires PyTorch and SciPy, but does not load pretrained weights or start training. NumPy, pandas, and Matplotlib support the comparison tables and figures.

In [ ]:
import json
import sys
from dataclasses import asdict, replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Make shared_code importable from stage4/ as well as the repository root.
working_directory = Path.cwd().resolve()
REPO_ROOT = next(
    folder for folder in (working_directory, *working_directory.parents)
    if (folder / "Code" / "shared_code" / "datasets.py").is_file()
)
CODE_DIR = REPO_ROOT / "Code"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from shared_code import datasets
from shared_code.encoders import ENCODERS
from shared_code.fusion import blf, token_embeddings, train
from shared_code.fusion.token_embeddings import (
    file_fingerprint, input_fingerprint, inputs_for_rows,
)

# A kernel can retain imports from another checkout even after sys.path changes.
for module in (datasets, blf, token_embeddings, train):
    if not Path(module.__file__).resolve().is_relative_to(CODE_DIR / "shared_code"):
        raise RuntimeError("Shared modules came from another checkout; restart this notebook's kernel.")

In [ ]:
BENCHMARK = "fungal_expression"
SEEDS = (0, 1, 2)
POSITIVE_LAMBDAS = (0.01, 0.5, 1.0)
RUN_TRAINING = False

# This notebook is a reader: changing the flag cannot accidentally launch fitting.
if RUN_TRAINING:
    raise RuntimeError("Training is not implemented here; this notebook only reads completed results.")

BASE_CONFIG = train.TrainConfig(
    lr=3e-5, weight_decay=1e-5, batch_size=32, max_epochs=500,
    early_stopping_patience=20, plateau_patience=5,
    channels=100, tconv="appendix", entropy_lambda=0.0, seed=0,
)
STAGE3_DIR = CODE_DIR / "stage3"
# Optional small headers: absence permits inspection, but blocks the final comparison.
CACHE_ROOT = STAGE3_DIR / "embeddings"
RESULT_DIRECTORIES = (STAGE3_DIR / "results" / BENCHMARK / "reproducible",)
# Choose a retained reference explicitly; never let glob order choose the producer cohort.
REFERENCE_RESULT_PATH = RESULT_DIRECTORIES[0] / (
    "fungal_expression__concat__lambda0.0__ch100__appendix__seed0.json"
)
COHORT_FIELDS = (
    "version", "model_kwargs", "widths", "n_outputs", "inputs", "environment", "code_sha256",
)
EXPECTED_WIDTHS = {"dna": 512, "rna": 640, "prot": 320}
EXPECTED_SPLITS = {"train": 2153, "val": 555, "test": 430}

# Declared order also breaks exact validation ties: DNA, RNA, protein; then lower lambda.
CANDIDATES = [
    ("DNA", "unimodal:dna", 0.0),
    ("RNA", "unimodal:rna", 0.0),
    ("Protein", "unimodal:prot", 0.0),
    ("Concat", "concat", 0.0),
    *[(f"MIL lambda={weight:g}", "mil", weight) for weight in POSITIVE_LAMBDAS],
    ("MIL without entropy", "mil", 0.0),
    ("Cross", "cross", 0.0),
]
plan = pd.DataFrame(
    [(seed, label, variant, weight) for seed in SEEDS for label, variant, weight in CANDIDATES],
    columns=["seed", "candidate", "variant", "entropy_lambda"],
)
KEY_COLUMNS = ["seed", "variant", "entropy_lambda"]
planned_keys = set(plan[KEY_COLUMNS].itertuples(index=False, name=None))
print(f"Training disabled. Planned comparison: {len(plan)} candidates across {len(SEEDS)} seeds.")

### What stays fixed, and what is selected?

All candidates use the same retained rows, supplied train/validation/test assignments, and frozen NT-v2 100M, RNA-FM, and ESM-2 8M embeddings. The fusion layers and prediction heads are trainable in the producing runs. For each seed, the candidate fits are the three single encoders, concatenation, the declared positive MIL entropy weights, MIL without entropy, and cross-attention.

The settings above follow the project's implementation of [BioLangFusion](../../Papers/core/BioLangFusion.pdf), Appendix A.4, List 3: Adam, MSE for training, and Spearman correlation for evaluation. The shared loop uses the validation score for scheduling, early stopping, and selecting the epoch whose test score is saved. The paper does not fully specify the monitored quantity. A seed labels a training realization; the same seed across architectures does not imply identical initial weights or dropout masks. This reader itself performs no random operations.

**Declared implementation choice.** The reference comparison uses the appendix transposed convolution and 100 channels. Equation 1 and the appendix differ on convolution settings, and the MIL listing uses a different channel width. Those alternatives belong in separate sensitivity comparisons, not a choice based on whichever test score is larger. Unimodal heads read their own native token resolution; the paper does not fully specify the corresponding baseline treatment. These are unresolved reproduction qualifications, not validated equivalences.

---

## Check the inputs behind the results

The shared loader strips and uppercases sequences, keeps rows at most 999 nucleotides long, and preserves the supplied split assignments. This check reads the CSV, not the embeddings. Counts alone cannot identify a dataset: the fingerprint below includes every retained sequence, length, target, and split in order, using the same serialization as `BenchmarkStreams.identity()`.

The expected retained counts are 2,153 training, 555 validation, and 430 test rows. The local raw file has 7,089 rows, whereas the paper reports 7,056. The local retention count of 3,138 agrees with the paper but does not prove identical examples. The paper describes truncation; this implementation filters out long rows. Target assay provenance, units, and transformation also require validation. [Step 1's dataset audit](../stage3/01_blf_datasets.ipynb) holds the detailed evidence.

In [ ]:
spec = datasets.DATASETS[BENCHMARK]
rows = datasets.blf_benchmark_rows(BENCHMARK)
split_counts = rows[spec.split_column].value_counts().to_dict()
if spec.task != "regression" or split_counts != EXPECTED_SPLITS:
    raise ValueError(f"Review the benchmark or split changes before comparing: {split_counts}")
if not np.isfinite(rows[spec.label_column].to_numpy(dtype=float)).all():
    raise ValueError("The retained target column contains nonfinite values.")

row_columns = ["sequence", "length", spec.label_column, spec.split_column]
row_texts = (
    json.dumps(row, ensure_ascii=False, separators=(",", ":"), default=lambda value: value.item())
    for row in rows[row_columns].itertuples(index=False, name=None)
)
rows_sha256 = input_fingerprint(row_texts)
display(pd.Series(split_counts, name="retained rows").reindex(EXPECTED_SPLITS).to_frame())

### Inspect the recorded inputs, and verify cache headers when available

A result records the source files, inputs, cache fingerprints, and environment of its producer. The reader can use those records after the code changes: the source currently reading a result is not the source that produced it. The explicitly selected reference above identifies the producer cohort used for comparison. Its source snapshot and verification history belong in the [run records](../../docs/run-records.md#2026-10-19-stage3-integration-preservation-of-completed-results-no-research-execution).

Each cache header identifies its encoder and ordered effective inputs. When headers are available under `CACHE_ROOT`, the next cell verifies those fields and records each header's hash. Completed records must match every available verified header. Missing or invalid headers remain visible below and block the final comparison; they do not prevent inspection of the recorded evidence. No cache is copied or created, and no embedding shard is opened.

Recorded cache and source fingerprints must be complete and consistent within the chosen cohort. This checks the records; it does not freshly verify large shard bytes, reproduce numerical results, or prove which modules a long-lived kernel had loaded. The environment reading these results may differ from the recorded producing environment.

In [ ]:
# Missing cache headers are a pending verification, not permission to invent their contents.
cache_headers = {}
cache_header_checks = []
for modality, encoder_name in blf.ENCODER_OF_MODALITY.items():
    metadata_path = CACHE_ROOT / BENCHMARK / encoder_name / "meta.json"
    if not metadata_path.is_file():
        cache_header_checks.append({
            "modality": modality, "status": "pending", "detail": "Cache header is unavailable.",
        })
        continue
    try:
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        expected_input = input_fingerprint(inputs_for_rows(ENCODERS[encoder_name], rows["sequence"]))
        if (metadata.get("encoder") != encoder_name
                or metadata.get("n_sequences") != len(rows)
                or metadata.get("input_sha256") != expected_input):
            raise ValueError("Header does not identify the current encoder inputs.")
        shard_count = metadata.get("n_shards")
        if type(shard_count) is not int or shard_count <= 0:
            raise ValueError("Expected a positive shard count.")
        cache_headers[modality] = {
            "meta_sha256": file_fingerprint(metadata_path),
            "filenames": {"meta.json", *(f"shard_{index:05d}.npz" for index in range(shard_count))},
        }
        cache_header_checks.append({
            "modality": modality, "status": "verified", "detail": "Header matches ordered encoder inputs.",
        })
    except (OSError, ValueError, TypeError, AttributeError) as error:
        cache_header_checks.append({"modality": modality, "status": "invalid", "detail": str(error)})

CACHE_HEADERS_VERIFIED = (
    set(cache_headers) == set(EXPECTED_WIDTHS)
    and all(check["status"] == "verified" for check in cache_header_checks)
)
display(pd.DataFrame(cache_header_checks).set_index("modality"))
print("Cache headers checked." if CACHE_HEADERS_VERIFIED
      else "Independent cache-header verification is pending or failed; final comparison stays blocked.")

# Require the producer's complete source list, without replacing it with this reader's hashes.
source_names = (
    "train.py", "blf.py", "codon_grid.py", "heads.py", "textcnn.py",
    "token_embeddings.py", "../datasets.py", "../sequences.py", "../encoders.py",
)

---

## Inspect completed runs for the declared comparison

Only final `.json` result files in `RESULT_DIRECTORIES` are read. Checkpoints and `.partial` files are not completed results; an absent JSON means **not available in this snapshot**, not a failed fit. No files are copied or rewritten. Rerun this section after more results finish to refresh the snapshot.

A represented candidate must match the declared variant, seed, entropy weight, full training configuration, current ordered rows, encoder widths, and output size. Its complete recorded cache, source, and environment identity must also match the explicit historical reference. The filename locates that reference; file contents determine compatibility. Legacy records without identity are excluded. Equation-convolution sensitivity runs remain separate from this appendix comparison, even when they are in the same directory.

The checks below separate record compatibility from independent artifact verification. Without cache headers, a compatible record can represent a planned fit, but it cannot make the final comparison ready. Each rejected file gets a reason so a malformed or unrelated record does not hide the rest of the inventory. The reference itself must pass the same checks.

In [ ]:
def check_protocol(result):
    """Require the declared experiment and complete recorded identity; return its grid key."""
    config = result.config
    key = (config["seed"], result.variant, config["entropy_lambda"])
    if type(config["seed"]) is not int or key not in planned_keys:
        raise ValueError("Variant, seed, or entropy weight is outside the planned grid.")
    expected_config = asdict(replace(BASE_CONFIG, seed=key[0], entropy_lambda=key[2]))
    if result.benchmark != BENCHMARK or config != expected_config:
        raise ValueError("Benchmark or full training configuration differs from the protocol.")
    identity = result.identity
    if not isinstance(identity, dict) or type(identity.get("version")) is not int or identity["version"] != 1:
        raise ValueError("Missing or unsupported run identity; legacy results are excluded.")
    json.dumps(identity, allow_nan=False)
    if identity.get("variant") != result.variant or identity.get("config") != config:
        raise ValueError("Result and embedded identity disagree.")
    if (identity.get("widths") != EXPECTED_WIDTHS or identity.get("n_outputs") != 1
            or identity.get("model_kwargs") != {}):
        raise ValueError("Model dimensions or arguments differ.")

    # These hashes describe the historical producer, not the currently imported reader.
    code_hashes = identity.get("code_sha256")
    if not isinstance(code_hashes, dict) or set(code_hashes) != set(source_names):
        raise ValueError("The recorded producer source list is incomplete or unexpected.")
    inputs = identity["inputs"]
    expected_inputs = {
        "benchmark": BENCHMARK, "task": "regression", "row_columns": row_columns,
        "rows_sha256": rows_sha256, "input_identity_status": "recorded_inputs",
    }
    if any(inputs.get(name) != value for name, value in expected_inputs.items()):
        raise ValueError("Ordered rows or input provenance differ from the current benchmark.")
    cache_files = inputs["cache_files"]
    if not isinstance(cache_files, dict) or set(cache_files) != set(EXPECTED_WIDTHS):
        raise ValueError("The recorded cache modalities are incomplete or unexpected.")
    fingerprint_groups = {"source": code_hashes}
    for modality, hashes in cache_files.items():
        if not isinstance(hashes, dict) or len(hashes) < 2:
            raise ValueError(f"{modality}: missing recorded cache fingerprints.")
        expected_files = {"meta.json", *(f"shard_{index:05d}.npz" for index in range(len(hashes) - 1))}
        if set(hashes) != expected_files:
            raise ValueError(f"{modality}: cache filenames must include metadata and contiguous shards.")
        if modality in cache_headers:
            header = cache_headers[modality]
            if set(hashes) != header["filenames"] or hashes["meta.json"] != header["meta_sha256"]:
                raise ValueError(f"{modality}: recorded cache files or header hash differ.")
        fingerprint_groups[modality] = hashes
    for name, hashes in fingerprint_groups.items():
        if any(not isinstance(value, str) or len(value) != 64
               or any(character not in "0123456789abcdef" for character in value)
               for value in hashes.values()):
            raise ValueError(f"{name}: missing or malformed SHA256 fingerprints.")

    environment = identity.get("environment")
    required_environment = {
        "python", "platform", "torch", "numpy", "scipy", "device", "threads", "interop_threads",
        "float32_matmul_precision", "deterministic_algorithms", "cudnn_deterministic",
        "cudnn_benchmark", "cudnn_allow_tf32", "cuda_matmul_allow_tf32", "cublas_workspace_config",
    }
    if isinstance(environment, dict) and str(environment.get("device")).startswith("cuda"):
        required_environment.update(("cuda", "device_name", "device_index", "cuda_device_count"))
    if not isinstance(environment, dict) or not required_environment <= environment.keys():
        raise ValueError("The producing environment record is incomplete.")
    if any(not isinstance(environment[name], str) or not environment[name]
           for name in ("python", "platform", "torch", "numpy", "scipy", "device")):
        raise ValueError("The producing environment has invalid version or device fields.")
    if any(type(environment[name]) is not int or environment[name] <= 0
           for name in ("threads", "interop_threads")):
        raise ValueError("The producing environment has invalid thread counts.")
    return key


def check_reference_cohort(result, reference_identity):
    """Keep the recorded producer, inputs, and environment together across candidates."""
    for field in COHORT_FIELDS:
        if result.identity[field] != reference_identity[field]:
            raise ValueError(f"Recorded {field} differs from the explicitly selected reference cohort.")

### Check completion and the selected epoch

`Result.load()` deserializes a file; it does not validate its contents. A usable result needs finite validation and test correlations in the range −1 to 1, a contiguous epoch history, and a saved validation score matching the first maximum in that history. Undefined early validation scores are allowed; a run with no defined validation score cannot select a model.

The stored `best_epoch` starts at zero. The displayed selection table adds one so that “epoch 1” means the first training epoch.

In [ ]:
def check_scores(result):
    """Reject incomplete or inconsistent scores before any validation selection."""
    for score in (result.val_score, result.test_score):
        if type(score) not in (int, float) or not np.isfinite(score) or not -1 <= score <= 1:
            raise ValueError("Validation and test Spearman must both be finite and in [-1, 1].")
    if type(result.epochs) is not int or not 1 <= result.epochs <= BASE_CONFIG.max_epochs:
        raise ValueError("Invalid completed epoch count.")
    if (len(result.history) != result.epochs
            or [entry["epoch"] for entry in result.history] != list(range(result.epochs))
            or type(result.best_epoch) is not int or not 0 <= result.best_epoch < result.epochs):
        raise ValueError("Epoch count, history, or selected epoch is inconsistent.")
    validation_history = [entry["val_score"] for entry in result.history]
    for score in validation_history:
        if score is not None and (type(score) not in (int, float)
                                  or not np.isfinite(score) or not -1 <= score <= 1):
            raise ValueError("Invalid validation score in the epoch history.")
    comparable = [-np.inf if score is None else score for score in validation_history]
    if (result.best_epoch != int(np.argmax(comparable))
            or result.val_score != comparable[result.best_epoch]):
        raise ValueError("Saved validation selection disagrees with the first best epoch.")

In [ ]:
# Snapshot final files once; another process may publish additional results after this cell.
result_paths = sorted({path for folder in RESULT_DIRECTORIES for path in folder.glob("*.json")})
inspected = []
excluded = []
reference_identity = None
try:
    reference_result = train.Result.load(REFERENCE_RESULT_PATH)
    check_protocol(reference_result)
    check_scores(reference_result)
    reference_identity = reference_result.identity
except (OSError, ValueError, TypeError, KeyError, AttributeError) as error:
    print(f"Reference cannot establish a cohort: {REFERENCE_RESULT_PATH.name}: {error}")

for path in result_paths:
    try:
        result = train.Result.load(path)
        seed, variant, entropy_weight = check_protocol(result)
        check_scores(result)
        if reference_identity is None:
            raise ValueError("The explicit reference is unavailable or invalid; cohort verification is blocked.")
        check_reference_cohort(result, reference_identity)
        inspected.append({
            "seed": seed, "variant": variant, "entropy_lambda": entropy_weight,
            "val_score": result.val_score, "test_score": result.test_score,
            "best_epoch": result.best_epoch + 1, "path": str(path.relative_to(REPO_ROOT)),
            "identity": result.identity,
            "has_test_truth": result.test_truth is not None,
            "has_test_prediction": result.test_prediction is not None,
        })
    except (OSError, ValueError, TypeError, KeyError, AttributeError) as error:
        excluded.append({"file": path.name, "reason": str(error)})

result_columns = [
    *KEY_COLUMNS, "val_score", "test_score", "best_epoch", "path", "identity",
    "has_test_truth", "has_test_prediction",
]
results = pd.DataFrame(inspected, columns=result_columns)
print(f"{len(result_paths)} completed files inspected; {len(results)} match the protocol and recorded reference cohort.")
print("Reference:", REFERENCE_RESULT_PATH.relative_to(REPO_ROOT))
if excluded:
    display(pd.DataFrame(excluded))

# Field presence is an inventory, not validation of prediction shapes or row correspondence.
prediction_availability = results.groupby(
    ["has_test_truth", "has_test_prediction"]
).size().rename("matching_records").reset_index()
print("Saved per-row fields among matching records (presence only):")
display(prediction_availability.rename(columns={
    "has_test_truth": "test targets stored",
    "has_test_prediction": "test predictions stored",
    "matching_records": "records",
}))

### Distinguish represented fits from a ready comparison

Every planned candidate needs exactly one compatible record. Duplicate records block comparison; neither file recency nor test performance resolves a duplicate. Runs must agree with the selected reference on recorded cache, source, and producing-environment identities. A different source or environment requires an explicit separate comparison, even when each record passes its individual checks. A future run produced by changed code does not automatically fill a gap in this historical cohort.

The table describes the inspected snapshot without selecting a winner from an incomplete lambda search. Each appendix fit represents only its own seed and candidate in the declared plan. Equation sensitivity fits remain outside that plan. Coverage is computed from the records, so a missing seed stays missing until compatible records for that seed are actually supplied.

**Represented** means one record passes the protocol, history, and reference-cohort checks. It does not mean the cache artifacts were independently verified. Every declared fit, one recorded cohort, and verified headers for every required modality are needed before selection, aggregate statistics, or figures. Missing headers and excluded records stay visible; nothing is retrained to repair a gap.

In [ ]:
counts = results.groupby(KEY_COLUMNS).size().rename("matching_files").reset_index()
coverage = plan.merge(counts, on=KEY_COLUMNS, how="left")
coverage["matching_files"] = coverage["matching_files"].fillna(0).astype(int)
coverage["status"] = coverage["matching_files"].map(
    lambda count: "missing" if count == 0 else "represented" if count == 1 else "duplicate"
)
display(coverage.pivot(index="candidate", columns="seed", values="status").reindex(
    [label for label, _, _ in CANDIDATES]
))

# Compare recorded identities, not this reader's code or current kernel environment.
cohort_signatures = results["identity"].map(
    lambda identity: json.dumps({field: identity[field] for field in COHORT_FIELDS}, sort_keys=True, allow_nan=False)
)
one_cohort = cohort_signatures.nunique() == 1
if cohort_signatures.nunique() > 1:
    print("Comparison blocked: more than one recorded input/code/environment cohort.")

GRID_COMPLETE = coverage["matching_files"].eq(1).all() and one_cohort
COMPARISON_READY = GRID_COMPLETE and CACHE_HEADERS_VERIFIED
print(f"{coverage['matching_files'].eq(1).sum()}/{len(plan)} planned candidates are represented exactly once.")
print("Independent cache-header verification:", "verified" if CACHE_HEADERS_VERIFIED else "pending or failed")
print("Ready for the declared comparison." if COMPARISON_READY
      else "Comparison pending: resolve missing, duplicate, incompatible, or unverified inputs.")

---

## Select on validation, then compare test scores

Within each seed, choose the positive MIL entropy weight with the highest validation Spearman. Separately choose the highest-validation single encoder as the comparator. Exact ties follow the order declared in Setup. MIL with zero entropy remains a separate ablation and never participates in the positive-lambda selection.

For fusion method $m$ and seed $s$, the paired difference is

$$
\Delta_{m,s} = \rho^{\mathrm{test}}_{m,s} - \rho^{\mathrm{test}}_{b_s,s},
\qquad
b_s = \underset{b\in\{\mathrm{DNA,RNA,Protein}\}}{\operatorname{argmax}}\;\rho^{\mathrm{val}}_{b,s}.
$$

Here $\rho$ is Spearman correlation: agreement between the ranking of predicted expression and the ranking of supplied targets. A positive $\Delta$ means the fusion procedure outscored the **validation-selected single encoder** for that seed. The baseline may differ between seeds; this evaluates a repeated selection procedure, not a comparison with one permanently fixed encoder.

The shared training loop already calculates test scores for every candidate. The code below does not use them to choose the lambda or baseline. Because the supplied test split has also been inspected during development, this is a reproduction comparison, not an untouched confirmatory evaluation.

In [ ]:
# Reset derived tables on every run so an incomplete refresh cannot retain an old comparison.
selected = pd.DataFrame()
choices = pd.DataFrame()
paired = pd.DataFrame()
MAIN_METHODS = ["DNA", "RNA", "Protein", "Concat", "MIL", "Cross"]
if COMPARISON_READY:
    ordered_results = plan.merge(results, on=KEY_COLUMNS, how="left", validate="one_to_one")
    selected_rows, choice_rows, difference_rows = [], [], []
    for seed in SEEDS:
        seed_results = ordered_results[ordered_results["seed"] == seed]
        single_runs = seed_results[seed_results["variant"].str.startswith("unimodal:")]
        positive_mil = seed_results[
            seed_results["variant"].eq("mil") & seed_results["entropy_lambda"].isin(POSITIVE_LAMBDAS)
        ]
        baseline = single_runs.loc[single_runs["val_score"].idxmax()]
        chosen_mil = positive_mil.loc[positive_mil["val_score"].idxmax()]
        choice_rows.append({
            "seed": seed, "single encoder": baseline["candidate"],
            "single validation": baseline["val_score"], "MIL lambda": chosen_mil["entropy_lambda"],
            "MIL validation": chosen_mil["val_score"],
        })
        # Fixed candidates plus the one validation-selected positive-lambda MIL run.
        reported_runs = pd.concat([seed_results[seed_results["entropy_lambda"].eq(0)], chosen_mil.to_frame().T])
        for _, run in reported_runs.iterrows():
            method = "MIL" if run["variant"] == "mil" and run["entropy_lambda"] > 0 else run["candidate"]
            selected_rows.append({"seed": seed, "method": method, "val_score": run["val_score"],
                                  "test_score": run["test_score"], "best_epoch": run["best_epoch"], "path": run["path"]})
            if method in ("Concat", "MIL", "Cross", "MIL without entropy"):
                difference_rows.append({"seed": seed, "method": method,
                                        "delta": run["test_score"] - baseline["test_score"]})
    selected = pd.DataFrame(selected_rows)
    choices = pd.DataFrame(choice_rows)
    paired = pd.DataFrame(difference_rows)
    display(choices.set_index("seed").round(4))
else:
    print("No model selection: the full planned grid is not ready.")

### Read the scores and their variation

The first table retains each seed, its validation and test score, and its selected epoch. The summary reports the arithmetic mean and sample standard deviation (`ddof=1`) of the three test correlations. These summarize optimization variation on this fixed split; they are not confidence intervals, biological replicates, or evidence of a significant gain.

The separate ablation table compares MIL without entropy with the same validation-selected single-encoder baseline. Do not silently replace the positive-lambda MIL procedure with this ablation if its test score is larger.

In [ ]:
if COMPARISON_READY:
    display(selected.drop(columns="path").sort_values(["method", "seed"]).round(4))
    summary = selected.groupby("method")["test_score"].agg(["count", "mean", "std"])
    display(summary.reindex(MAIN_METHODS).rename(columns={"count": "seeds", "std": "seed SD"}).round(4))
    ablation = selected[selected["method"] == "MIL without entropy"].merge(
        paired, on=["seed", "method"], validate="one_to_one"
    )
    display(ablation[["seed", "val_score", "test_score", "delta"]].round(4))
else:
    print("Score summaries wait for a complete, comparable grid.")

### Test performance and paired gains

The first figure shows all three seeds and their mean for each main method. The second subtracts each seed's validation-selected single-encoder score from its fusion score. The zero line marks equal performance. Points use the same seed colors across both figures; with only three seeds, show the observations directly rather than implying a distribution with a boxplot or confidence band.

Compare the pattern with [BioLangFusion's Table 1](../../Papers/core/BioLangFusion.pdf), keeping the dataset and implementation differences above in view. Numerical closeness to a paper value alone does not validate a reproduction. These plots use existing aggregate scores. The inventory above reports whether matching records store per-example test targets and predictions. Aggregate scores alone cannot supply residual plots, prediction scatterplots, or row-bootstrap intervals. The presence of both fields is only a prerequisite: their shapes, numerical values, and correspondence to the same test rows still need validation before such analyses. This reader does not load model weights or perform prediction-level analysis.

In [ ]:
if COMPARISON_READY:
    score_matrix = selected.pivot(index="method", columns="seed", values="test_score").reindex(MAIN_METHODS)
    positions = np.arange(len(MAIN_METHODS))
    offsets = np.linspace(-0.15, 0.15, len(SEEDS))
    seed_colors = ("#0072B2", "#D55E00", "#009E73")
    figure, axis = plt.subplots(figsize=(9, 4.5), layout="constrained")
    for seed, offset, color in zip(SEEDS, offsets, seed_colors):
        axis.scatter(positions + offset, score_matrix[seed], label=f"Seed {seed}", color=color, s=48)
    axis.scatter(positions, score_matrix.mean(axis=1), marker="_", s=250, color="black", label="Mean", zorder=3)
    axis.set(xticks=positions, xticklabels=MAIN_METHODS, ylabel="Test Spearman correlation",
             title="Fungal expression: three training seeds on the supplied split")
    axis.grid(axis="y", alpha=0.25)
    axis.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.12))
    plt.show()
else:
    print("Performance figure pending the complete grid.")

In [ ]:
if COMPARISON_READY:
    fusion_methods = ["Concat", "MIL", "Cross"]
    delta_matrix = paired.pivot(index="method", columns="seed", values="delta").reindex(fusion_methods)
    positions = np.arange(len(fusion_methods))
    figure, axis = plt.subplots(figsize=(8, 4.5), layout="constrained")
    for seed, offset, color in zip(SEEDS, offsets, seed_colors):
        axis.scatter(positions + offset, delta_matrix[seed], label=f"Seed {seed}", color=color, s=48)
    axis.scatter(positions, delta_matrix.mean(axis=1), marker="_", s=250, color="black", label="Mean", zorder=3)
    axis.axhline(0, color="gray", linestyle="--", linewidth=1)
    axis.set(xticks=positions, xticklabels=fusion_methods, ylabel="Difference in test Spearman",
             title="Fusion minus the validation-selected single encoder, within each seed")
    axis.grid(axis="y", alpha=0.25)
    axis.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.12))
    plt.show()
else:
    print("Paired-difference figure pending the complete grid.")

---

## Interpret before extending the experiment

A complete grid makes the specified comparison readable; it does not settle faithful paper reproduction or biological validity. Review whether each fusion gain is consistent across the three seeds, which positive MIL weight validation selects, and whether the zero-entropy ablation changes the interpretation. A fusion gain concerns complementary predictive representations of the supplied sequence, not independent molecular evidence.

Before stronger claims, resolve the retained-row correspondence and filtering-versus-truncation issue, trace the target assay and transformations, and review the convolution, channel-width, and unimodal-baseline choices. The NT cache has 512 channels; the paper's 4,107-feature description also needs reconciliation. The dataset audit reports no exact-sequence overlap across the retained splits, but that does not establish independence from homologous sequences, shared composition, or encoder pretraining data.

Run lineage belongs in the completed result identities and [run records](../../docs/run-records.md), including producing code/commit, local differences, environment, and retained inputs. Source fingerprints alone cannot establish which module objects a long-lived kernel executed. When reporting this comparison, identify the exact result paths stored in `selected` and retain the validation choices. This notebook does not write a run record or export artifacts automatically.

To complete a pending grid, specify the missing candidate fits and their result directory in the execution workflow. This notebook collects those results after they are produced. Encoder swaps, new fusion designs, additional benchmarks, and prediction-level uncertainty are separate extensions after this comparison is reviewed. See the [Stage 4 plan](README.md) and [Gantt workbook](../../docs/Research_Project_Plan_Gantt.xlsx), `Details` items 17, 19, and 24.